# M11 - Worksheet LoRA\nRetrain the tutor adapter on 706 original dialogues + 402 generated from the\n12 real Year-2 worksheets (ground truth from teacher marks / Python-verified).\nHeld-out: 3 whole worksheets (191 questions) - never seen in training.\n

In [ ]:

# ---- C1 setup: paths, resume helpers, shared constants + classes -----------
import json, os, sys, time, types, collections, traceback, hashlib
from pathlib import Path

import mlx.core as mx
from mlx.utils import tree_flatten, tree_unflatten

# jupyter execute may set the kernel cwd to the notebook dir; resolve root robustly.
ROOT = Path.cwd()
if not (ROOT / "PROJECT_BRIEF.md").exists():
    for cand in (ROOT, ROOT.parent, ROOT.parent.parent):
        if (cand / "PROJECT_BRIEF.md").exists():
            ROOT = cand
            break
os.chdir(ROOT)
assert (ROOT / "PROJECT_BRIEF.md").exists(), "cannot locate PROJECT_BRIEF.md from cwd=" + str(Path.cwd())

MLX_DIR = "models/Qwen3.5-4B-MLX-8bit"
HF_DIR = "models/Qwen3.5-4B"
GEETA_MLX_DIR = "models/geeta-4b-mlx"
GEETA_HF_DIR = "models/geeta-4b-hf"
GEETA_GGUF_DIR = "models/geeta-4b"
LLAMA_CPP = "third_party/llama.cpp"
OUT = ROOT / "outputs" / "m4"
OUT.mkdir(parents=True, exist_ok=True)

SENT = chr(10) + "image" + chr(10)              # 7-char chat-template sentinel
ASSIST_OPEN = chr(96) + "s" + chr(105) + chr(115) + chr(116) + "ant"
SYS_OPEN = chr(115) + chr(121) + "stem"

def done(flag):
    return (OUT / ("." + flag + ".done")).exists()

def mark(flag, payload=None):
    p = OUT / ("." + flag + ".done")
    if not p.exists():
        p.write_text(json.dumps({"flag": flag, "ts": time.time(), "payload": payload}) + chr(10))
    print("[m4] flagged", flag, "done")

def log(name, payload):
    path = OUT / name
    path.write_text(json.dumps(payload, indent=2, default=str))
    print("[m4] wrote", path)

def mem():
    return "peak=" + str(round(mx.get_peak_memory() / 2**30, 1)) + "GiB"

# --- shared chat dataset: exact per-row prefix/offset, enable_thinking=False --
class GeetaChatDataset:
    def __init__(self, rows, tokenizer, max_seq_length=2048):
        self.tokenizer = tokenizer
        self.max_seq_length = max_seq_length
        self.rows = rows
    def process(self, row):
        messages = row["messages"]
        agp = messages[-1]["role"] == "user"
        tokens = self.tokenizer.apply_chat_template(messages, tools=None,
            add_generation_prompt=agp, enable_thinking=False, return_dict=False)
        if len(messages) > 1:
            prefix = self.tokenizer.apply_chat_template(messages[:-1], tools=None,
                add_generation_prompt=True, enable_thinking=False, return_dict=False)
        else:
            prefix = []
        return list(tokens), len(prefix)
    def __getitem__(self, idx):
        return self.rows[idx]
    def __len__(self):
        return len(self.rows)

def expand(rows):
    """Prefix-expand each dialogue into one row per assistant turn (loss = that turn)."""
    out = []
    for r in rows:
        msgs = r["messages"]
        n_asst = sum(1 for m in msgs if m["role"] == "assistant")
        for i in range(n_asst):
            out.append({"messages": msgs[: 2 * i + 2], "source": r.get("source"),
                        "role": r.get("role"), "expansion_of": r.get("id")})
    return out

class LossCapture:
    def __init__(self):
        self.train = []
        self.val = []
    def on_train_loss_report(self, info):
        self.train.append({"it": info["iteration"], "loss": round(info["train_loss"], 4),
                           "lr": info["learning_rate"], "tok_s": round(info["tokens_per_second"], 1)})
    def on_val_loss_report(self, info):
        self.val.append({"it": info["iteration"], "loss": round(info["val_loss"], 4)})

def run_lora_training(model, tokenizer, train_rows, valid_rows, *,
                      num_layers=16, lora_params=None, batch_size=1, iters=100,
                      val_batches=5, learning_rate=1e-5, lr_schedule=None,
                      max_seq_length=2048, grad_checkpoint=True, grad_accum=1,
                      steps_per_report=10, steps_per_eval=50, save_every=None,
                      adapter_path="outputs/m4/lora", seed=0):
    """Reproduce mlx_lm.lora.train_model() plumbing with our exact-offset datasets."""
    import mlx.nn as nn
    import mlx.optimizers as optim
    from mlx_lm.tuner.datasets import CacheDataset
    from mlx_lm.tuner.trainer import TrainingArgs, train
    from mlx_lm.tuner.utils import linear_to_lora_layers, build_schedule
    from mlx_lm.utils import save_config

    if lora_params is None:
        lora_params = {"rank": 16, "scale": 2.0, "dropout": 0.0}
    adapter_path = Path(adapter_path)
    adapter_path.mkdir(parents=True, exist_ok=True)

    model.freeze()
    linear_to_lora_layers(model, num_layers, lora_params)
    n_train = sum(p.size for _, p in tree_flatten(model.trainable_parameters()))
    n_total = sum(p.size for _, p in tree_flatten(model.parameters()))
    print("[lora] trainable %.1fM / %.1fM (%.2f%%)" % (n_train/1e6, n_total/1e6, 100*n_train/max(n_total,1)))

    args_ns = types.SimpleNamespace(
        model=MLX_DIR, train=True, fine_tune_type="lora", optimizer="adam",
        data=str(adapter_path), seed=seed, num_layers=num_layers, batch_size=batch_size,
        iters=iters, val_batches=val_batches, learning_rate=learning_rate,
        steps_per_report=steps_per_report, steps_per_eval=steps_per_eval,
        resume_adapter_file=None, adapter_path=str(adapter_path), save_every=save_every or iters,
        test=False, test_batches=500, max_seq_length=max_seq_length, config=None,
        grad_checkpoint=grad_checkpoint, grad_accumulation_steps=grad_accum,
        clear_cache_threshold=0, lr_schedule=lr_schedule, lora_parameters=lora_params,
        mask_prompt=True, report_to=None, project_name="geeta")
    save_config(vars(args_ns), adapter_path / "adapter_config.json")

    training_args = TrainingArgs(
        batch_size=batch_size, iters=iters, val_batches=val_batches,
        steps_per_report=steps_per_report, steps_per_eval=steps_per_eval,
        steps_per_save=save_every or iters, adapter_file=str(adapter_path / "adapters.safetensors"),
        max_seq_length=max_seq_length, grad_checkpoint=grad_checkpoint,
        grad_accumulation_steps=grad_accum)

    lr = build_schedule(lr_schedule) if lr_schedule else learning_rate
    opt = optim.Adam(learning_rate=lr)

    train_set = GeetaChatDataset(train_rows, tokenizer, max_seq_length=max_seq_length)
    valid_set = GeetaChatDataset(valid_rows, tokenizer, max_seq_length=max_seq_length) if valid_rows else None

    cb = LossCapture()
    train(model=model, args=training_args, optimizer=opt,
          train_dataset=CacheDataset(train_set),
          val_dataset=CacheDataset(valid_set) if valid_set else None,
          training_callback=cb)

    mx.save_safetensors(str(adapter_path / "adapters.safetensors"),
                        dict(tree_flatten(model.trainable_parameters())))
    return cb

print("C1 ready | root:", str(ROOT))


In [ ]:
# ---- Worksheet LoRA: 706 original + 402 real-worksheet dialogues -----------
import mlx_lm, hashlib, json, time

TRAIN_FILE = "data/splits/train_with_worksheets.jsonl"
ADAPTER = OUT / "worksheet_lora"

rows = [json.loads(l) for l in open(TRAIN_FILE)]
for r in rows:
    r.setdefault("id", hashlib.sha256(json.dumps(r["messages"], sort_keys=True).encode()).hexdigest()[:16])

n_valid = 30
valid_d, train_d = rows[-n_valid:], rows[:-n_valid]
train_exp, valid_exp = expand(train_d), expand(valid_d)
(OUT / "worksheet_lora_data").mkdir(parents=True, exist_ok=True)
for name, data in (("train", train_exp), ("valid", valid_exp)):
    with open(OUT / "worksheet_lora_data" / f"{name}.jsonl", "w") as f:
        for r in data:
            f.write(json.dumps(r) + chr(10))
print("data: %d train rows (%d dialogues) | %d valid rows (%d dialogues)"
      % (len(train_exp), len(train_d), len(valid_exp), n_valid), flush=True)

model, tokenizer = mlx_lm.load(MLX_DIR)
iters = len(train_exp)
updates = iters // 2
lr_schedule = {"name": "cosine_decay", "arguments": [1e-5, updates, 1e-6],
               "warmup": 50, "warmup_init": 1e-6}
print("iters=%d (1 epoch) grad_accum=2 -> %d updates" % (iters, updates), flush=True)

t0 = time.time()
cb = run_lora_training(
    model, tokenizer, train_exp, valid_exp,
    num_layers=16, lora_params={"rank": 16, "scale": 2.0, "dropout": 0.0},
    batch_size=1, iters=iters, val_batches=10, learning_rate=1e-5, lr_schedule=lr_schedule,
    max_seq_length=2048, grad_checkpoint=True, grad_accum=2,
    steps_per_report=50, steps_per_eval=200, save_every=300,
    adapter_path=str(ADAPTER), seed=0)
dt = time.time() - t0

vfirst = cb.val[0]["loss"] if cb.val else None
vlast = cb.val[-1]["loss"] if cb.val else None
log("worksheet_lora_report.json", {
    "train_file": TRAIN_FILE, "n_dialogues": len(rows), "n_train_rows": len(train_exp),
    "iters": iters, "minutes": round(dt / 60, 1),
    "val_loss_first": vfirst, "val_loss_last": vlast,
    "val_improved": (vfirst is not None and vlast is not None and vlast < vfirst),
    "adapter": str(ADAPTER), "train_curve": cb.train[-20:], "val_curve": cb.val})
print("DONE in %.1f min | val loss %s -> %s" % (dt / 60, vfirst, vlast), flush=True)
